In [1]:
import os
import urllib.request

import numpy as np
import pytest
import scanpy as sc
import torch
from scdataloader import Preprocessor
from scdataloader.utils import populate_my_ontology

from scprint import scPrint
from scprint.base import NAME
from scprint.tasks import Denoiser, Embedder, GNInfer

%reload_ext autoreload
%autoreload 2


→ connected lamindb: zhang0730/lamin_data


In [2]:
adata = sc.read_h5ad("test.h5ad")
adata.obs.drop(columns="is_primary_data", inplace=True, errors="ignore")
adata.obs["organism_ontology_term_id"] = "NCBITaxon:9606"

In [3]:
adata.obs

,biosample_id,donor_id,cell_type_ontology_term_id,organism_ontology_term_id,disease_ontology_term_id,tissue_ontology_term_id,assay_ontology_term_id,cell_type__custom,development_stage_ontology_term_id,sex_ontology_term_id,suspension_type,age,self_reported_ethnicity_ontology_term_id,cell_type,assay,disease,organism,sex,tissue,self_reported_ethnicity,development_stage,cell_culture,nnz,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb,outlier,mt_outlier
c14eb8e2-c476-463e-aef9-b5f8eecba1a8,LVG1_retina_OD,LVG1,CL:0000740,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,Retinal ganglion cell,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,retinal ganglion cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,5449,5442,8.602086,15434.0,9.644393,13.120384,3.0,1.386294,0.019438,234.0,5.459586,1.516133,4.0,1.609438,0.025917,True,False
b8575573-aae6-48e5-ba11-9f6d5bbf512f,LVG1_retina_OD,LVG1,CL:0000573,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,Cone,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,retinal cone cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,5373,5366,8.588024,19278.0,9.866772,13.248262,8.0,2.197225,0.041498,298.0,5.700444,1.545804,1.0,0.693147,0.005187,True,True
0119aa9e-562e-439d-9d2a-d83298ebec51,LVG1_retina_OD,LVG1,CL:0000561,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,AII-amacrine,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,amacrine cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,8046,8042,8.992557,29465.0,10.290993,11.030036,5.0,1.791759,0.016969,631.0,6.448889,2.141524,2.0,1.098612,0.006788,True,False
f5e9b03a-dd93-4cfb-8350-65e2a37f1357,LVG1_retina_OD,LVG1,CL:0000561,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,GABA-amacrine,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,amacrine cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,5320,5312,8.577912,17468.0,9.768183,12.852072,3.0,1.386294,0.017174,175.0,5.170484,1.001832,0.0,0.000000,0.000000,True,False
f82f58df-83fe-4496-8116-6e4c67829c89,LVG1_retina_OD,LVG1,CL:0000740,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,Retinal ganglion cell,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,retinal ganglion cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,5751,5741,8.655563,16519.0,9.712327,11.338459,0.0,0.000000,0.000000,277.0,5.627621,1.676857,3.0,1.386294,0.018161,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444d1ca6-c98c-420a-a69a-d988d31f79a5,LVG1_retina_OD,LVG1,CL:0000636,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,Muller glia,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,Mueller cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,2207,2206,7.699389,4194.0,8.341649,18.621841,1.0,0.693147,0.023844,42.0,3.761200,1.001431,0.0,0.000000,0.000000,False,False
ec554ff9-b91a-44be-aa1d-e81cdb94341e,LVG1_retina_OD,LVG1,CL:0000636,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,Muller glia,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,Mueller cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,2334,2331,7.754482,4813.0,8.479283,16.143777,2.0,1.098612,0.041554,80.0,4.394449,1.662165,0.0,0.000000,0.000000,False,True
44d492e5-e9d7-49b5-bc92-6cae86aec05d,LVG1_retina_OD,LVG1,CL:0000751,NCBITaxon:9606,PATO:0000461,UBERON:0000966,EFO:0030059,Rod bipolar,HsapDv:0000149,PATO:0000384,nucleus,55,unknown,rod bipolar cell,10x multiome,normal,Homo sapiens,male,retina,unknown,55-year-old human stage,False,2069,2068,7.634821,4351.0,8.378

In [4]:
preprocessor = Preprocessor(
    do_postp=False,
    force_preprocess=True,
)
adata = preprocessor(adata)

Dropping layers:  KeysView(Layers with keys: )
checking raw counts
removed 0 non primary cells, 1000 renamining
filtered out 0 cells, 1000 renamining
Removed 0 genes.
validating


/disk2/044/anaconda3/envs/scprint/lib/python3.10/site-packages/scdataloader/preprocess.py:272: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  adata, organism=adata.obs.organism_ontology_term_id[0], need_all=False


startin QC
Seeing 161 outliers (16.10% of total dataset):
done
AnnData object with n_obs × n_vars = 1000 × 70611
    obs: 'biosample_id', 'donor_id', 'cell_type_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'cell_type__custom', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'age', 'self_reported_ethnicity_ontology_term_id', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'cell_culture', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_genes'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotyp

In [5]:
ckpt_path = "small.ckpt" #缺少了这个文件，并且一直找不到
model = scPrint.load_from_checkpoint(
    ckpt_path,
    precpt_gene_emb=None,
    # triton gets installed so it must think it has cuda enabled
    transformer="normal",
)

FileNotFoundError: [Errno 2] No such file or directory: '/disk2/044/scprint/scPRINT-main/tests/small.ckpt'

In [20]:
adata

View of AnnData object with n_obs × n_vars = 1000 × 70611
    obs: 'biosample_id', 'donor_id', 'cell_type_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'cell_type__custom', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'age', 'self_reported_ethnicity_ontology_term_id', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'cell_culture', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_genes'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'uid', 'symbol', 'ncbi_gene_ids', 'biotype', 'descr

In [21]:
cell_embedder = Embedder(
    batch_size=6,
    num_workers=1,
    how="random expr",
    max_len=300,
    doclass=True,
    pred_embedding=[
        "cell_type_ontology_term_id",
        "disease_ontology_term_id",
        "self_reported_ethnicity_ontology_term_id",
        "sex_ontology_term_id",
    ],
    plot_corr_size=10,
    doplot=True,
    keep_all_cls_pred=False,
    dtype=torch.float32,
)
adata_emb, metrics = cell_embedder(model, adata[:10, :])

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


View of AnnData object with n_obs × n_vars = 10 × 70611
    obs: 'biosample_id', 'donor_id', 'cell_type_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'cell_type__custom', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'age', 'self_reported_ethnicity_ontology_term_id', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'cell_culture', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_genes'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'uid', 'symbol', 'ncbi_gene_ids', 'biotype', 'descrip

 50%|█████     | 1/2 [00:00<00:00,  7.37it/s]

hey
hey


100%|██████████| 2/2 [00:00<00:00,  8.71it/s]


AnnData object with n_obs × n_vars = 10 × 128
    obs: 'pred_cell_type_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 'pred_sex_ontology_term_id', 'pred_organism_ontology_term_id', 'conv_pred_cell_type_ontology_term_id', 'conv_pred_disease_ontology_term_id', 'conv_pred_assay_ontology_term_id', 'conv_pred_self_reported_ethnicity_ontology_term_id'
couldn't log to tensorboard
couldn't log to wandb
AnnData object with n_obs × n_vars = 10 × 128
    obs: 'pred_cell_type_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 'pred_sex_ontology_term_id', 'pred_organism_ontology_term_id', 'conv_pred_cell_type_ontology_term_id', 'conv_pred_disease_ontology_term_id', 'conv_pred_assay_ontology_term_id', 'conv_pred_self_reported_ethnicity_ontology_term_id'
too few cells to embed into a umap
too few cells to compute a clustering
     cell_t

In [22]:
adata_emb

AnnData object with n_obs × n_vars = 10 × 70611
    obs: 'biosample_id', 'donor_id', 'cell_type_ontology_term_id', 'organism_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'cell_type__custom', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'age', 'self_reported_ethnicity_ontology_term_id', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'cell_culture', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_genes', 'pred_cell_type_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 

In [24]:
assert any(
    col.startswith("pred_") for col in adata_emb.obs.columns
), "Classification failed"

In [25]:
grn_inferer = GNInfer(
    layer=[0, 1],
    batch_size=2,
    how="random expr",
    preprocess="softmax",
    head_agg="mean",
    filtration="none",
    forward_mode="none",
    num_genes=100,
    max_cells=10,
    doplot=False,
)
grn_adata = grn_inferer(model, adata)

100%|██████████| 5/5 [00:00<00:00,  7.66it/s]


avg link count: 85969984, sparsity: 1.0


In [27]:
assert "GRN" in grn_adata.varp, "GRN inference failed"